# GroundingDINO quick test

Run GroundingDINO on a single image to tune the prompt and the thresholds.

In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.detection import GroundingDINODetector

In [ ]:
MASK_COLORS = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0), (255, 0, 255)]


def draw_boxes(image, detections, color=(0, 255, 0)):
    """Draw detections on a BGR image, line and font sizes scaled to the image."""
    img_copy = image.copy()
    h, w = img_copy.shape[:2]

    scale_factor = max(h, w) / 1000.0
    thickness = max(1, int(2 * scale_factor))
    font_scale = max(0.4, 0.6 * scale_factor)
    font_thickness = max(1, int(1.5 * scale_factor))
    padding = max(2, int(5 * scale_factor))

    for det in detections:
        x1, y1, x2, y2 = map(int, det['bbox'])
        cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, thickness)

        text = f"{det['label']}: {det['confidence']:.2f}"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, font_scale, font_thickness)

        label_y = y1 - padding
        if label_y - th - padding < 0:
            label_y = y2 + th + padding

        cv2.rectangle(img_copy,
                      (x1, label_y - th - padding),
                      (x1 + tw + padding, label_y + padding // 2),
                      color, -1)
        cv2.putText(img_copy, text,
                    (x1 + padding // 2, label_y - padding // 2),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale, (0, 0, 0), font_thickness, cv2.LINE_AA)

    return img_copy


def draw_masks(image, masks, alpha=0.5):
    img_copy = image.copy()
    for i, mask in enumerate(masks):
        colored_mask = np.zeros_like(img_copy)
        colored_mask[mask > 0] = MASK_COLORS[i % len(MASK_COLORS)]
        img_copy = cv2.addWeighted(img_copy, 1, colored_mask, alpha, 0)
    return img_copy


def visualize_detections(image_path, detections, masks=None):
    vis_img = cv2.imread(str(image_path))
    if masks:
        vis_img = draw_masks(vis_img, masks, alpha=0.4)
    vis_img = draw_boxes(vis_img, detections, color=(0, 255, 0))

    fig = plt.figure(figsize=(20, 15))
    plt.imshow(cv2.cvtColor(vis_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(image_path.name, fontsize=16, fontweight='bold')
    plt.tight_layout()
    return fig

In [ ]:
def quick_test_gdino(image_path, prompt, box_threshold=0.35, text_threshold=0.25):
    """Detect prompt ("red ball . person") on one image and plot the result."""
    img_path = Path(image_path)
    detector = GroundingDINODetector(box_threshold=box_threshold, text_threshold=text_threshold)
    dets_dict = detector.detect(img_path, prompt, return_all_by_label=True)

    print(", ".join(f"{len(dets)} {label}" for label, dets in dets_dict.items()))

    all_dets = [det for dets in dets_dict.values() for det in dets]
    return visualize_detections(img_path, all_dets)

In [ ]:
img = Path("datasets/raw/red balls human yahoo/OIP.0bNjCsWeOxnH2t-3VE75TAHaE7.jpg")
assert img.exists()
quick_test_gdino(img, "red ball . person . trashcan", box_threshold=0.116, text_threshold=0.2)

In [ ]:
img = Path("datasets/raw/google dated_converted_jpg/trashcans_lens/qjn7MpC.jpg")
assert img.exists()
quick_test_gdino(img, "human . trashcan", box_threshold=0.45, text_threshold=0.2)

In [ ]:
img = Path("datasets/raw/red balls human yahoo/OIP.0QEG5dHl1XzqFM3XaQcucgAAAA.jpg")
assert img.exists()
quick_test_gdino(img, "red ball . person", box_threshold=0.116, text_threshold=0.2)